# 기준선 비교 — TF-IDF + 선형 모델

라벨이 확정됐으니 이제 **"복잡하게 만든 값을 했는가"**를 묻기 시작합니다(§9).
그 질문에 답하려면 먼저 **가장 단순한 것이 어디까지 하는지** 알아야 합니다.

이 노트북은 그 출발점입니다. 파인튜닝은 아직 하지 않습니다.

## 무엇을 보는가

1. **fold를 어떻게 읽어야 하는가** — 점수를 내기 전에 fold마다 난이도와 점수 구성이
   얼마나 다른지 확인합니다. 이걸 모르고 점수를 보면 모델이 아니라 분할을 보게 됩니다.
2. **기준선 6종** — Dummy부터 LinearSVC와 검토 가중치까지. 각 단계가 얼마나 더 하는가.
3. **통제 비교** — 한 번에 하나만 바꿔서 무엇이 효과였는지 가려냅니다.
4. **macro F1 해부** — 왜 이 지표를 쓰는지, 같은 예측이 지표에 따라 어떻게 달리 보이는지.
5. **세 갈래 점수** — 표준 문구 반복이 점수에 얼마나 섞여 있는지(결정 34).

## 쓰는 데이터와 분할

**확정(동결) 데이터셋** `label_dataset_v3.jsonl` 1,024건, 문서 단위 **LODO 10겹**입니다.
이미 라벨 예시로 쓰인 동결 앵커 100건은 검증·평가에서 제외해 평가 합계는 924건입니다.
fold마다 학습 8문서 / 검증 1문서 / 평가 1문서로 가릅니다(2026-08-23 결정).

실제 로직은 `scripts/evaluation/`에 있습니다. 이 노트북은 **사용법과 결과 해석**만 담습니다.
각 모듈의 docstring에 "왜 그렇게 하는가"가 적혀 있으니 함께 읽으면 좋습니다.

- `folds.py` — fold 생성과 fold별 진단
- `duplication.py` — 문서 간 표준 문구 반복 측정
- `baselines.py` — 지표 계산, LODO 실행, 통제 비교

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / 'scripts').is_dir() and (p / 'data').is_dir()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(ROOT))

installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next(
    f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'AppleGothic', 'DejaVu Sans']
    if f in installed
)
plt.rcParams['axes.unicode_minus'] = False

from scripts.evaluation.baselines import (
    CHAR_BALANCED, CHAR_UNWEIGHTED, DUMMY, SVM_BALANCED, WORD_BALANCED,
    run_lodo, run_review_weight_tuned_lodo, summarize,
)
from scripts.evaluation.folds import diagnose_all, make_lodo_folds
from scripts.labeling.label_dataset import load_label_dataset

rows, meta = load_label_dataset()
print(f"데이터셋 {meta['dataset_version']} / {meta['row_count']}건 / 문서 {meta['document_count']}개")
print(f"sha256   {meta['sha256'][:16]}...")

display(Markdown('''## 1. 점수를 내기 전에 — fold를 어떻게 읽어야 하는가

fold 점수는 **그 자체로 비교할 수 없습니다.** 두 가지가 fold마다 다르기 때문입니다.

- **난이도** — 다수 클래스만 찍어도 koen은 79.2%, defense는 34.3%입니다(issues/003)
- **점수 구성** — 실제 학습 8문서 기준 반복 노출이 mfds 23.8%, ccrs 0.0%입니다(issues/006)

`Dummy`는 학습 최빈을 찍는 **배포 가능한** 모델이고, `oracle`은 평가 문서 안의 최빈
비율이라 정답을 미리 본 값입니다. 문서마다 최빈 클래스가 달라 둘이 갈리는 fold가 있습니다.
'''))

diag = pd.DataFrame([{
    '평가 문서': d.fold.test_document,
    '건수': d.test_size,
    'Dummy': d.trained_majority_accuracy,
    'oracle': d.oracle_majority_accuracy,
    '반복노출': d.repeat_exposure_rate,
    '검증 문서': d.fold.validation_document,
} for d in diagnose_all(rows)]).set_index('평가 문서')
display(diag.style.format({'Dummy': '{:.1%}', 'oracle': '{:.1%}', '반복노출': '{:.1%}'}))

r = diag['oracle'].corr(diag['반복노출'])
print(f"oracle 기준선과 반복 노출률의 상관 r = {r:.3f}")
print("003과 006은 독립이 아닙니다. 쉬운 fold가 표준 문구라는 공짜 점수까지 더 받습니다.")
print("따로 할인하면 이중 할인, 하나만 보면 과대평가가 됩니다.")

In [ ]:
display(Markdown('''## 2. 기준선 6종 — 각 단계가 얼마나 더 하는가

아래로 갈수록 복잡해집니다. **각 줄은 바로 윗줄이 못한 무엇을 하는지**로 존재 이유가 있습니다.

| 설정 | 무엇을 확인하는가 |
|---|---|
| `Dummy` | 본문을 아예 안 보면 몇 점인가 (§9.2의 최저 기준) |
| `word 1-2gram` | 단어 겹침만으로 얼마나 되는가 |
| `char 3-4gram, weight 없음` | 문자 n-gram으로 바꾸면 |
| `char 3-4gram + balanced` | 불균형 보정까지 켜면 |
| `LinearSVC + balanced` | 같은 표현에서 분류기만 바꾸면 |
| `LinearSVC + 검증 weight` | 검토 recall을 더 중시하면 |

`Dummy대비`는 정확도에서 fold별 Dummy를 뺀 값입니다. **fold 간 절대 점수는 비교할 수
없지만 기준선 대비 개선폭은 비교할 수 있습니다**(issues/003).

검토 가중치 후보는 `1.0 / 1.25 / 1.5 / 2.0`으로 고정합니다. 각 outer fold의 학습
8문서에서만 balanced 가중치를 계산하고, 회전 검증 1문서의 계약 F2로 후보를 고릅니다.
동률이면 macro F1, 더 작은 배수 순입니다. 평가 문서는 선택에 사용하지 않습니다.
'''))

specs = [DUMMY, WORD_BALANCED, CHAR_UNWEIGHTED, CHAR_BALANCED, SVM_BALANCED]
results = {s.name: run_lodo(rows, s) for s in specs}
results['LinearSVC + 검증 weight'] = run_review_weight_tuned_lodo(rows, SVM_BALANCED)

ladder = pd.DataFrame([{
    '설정': name,
    'macroF1': summarize(res)['macro_f1']['fold_mean'],
    'macroF1(건수가중)': summarize(res)['macro_f1']['count_weighted'],
    '정확도': summarize(res)['accuracy']['fold_mean'],
    '계약precision': summarize(res)['review_precision']['fold_mean'],
    '계약recall': summarize(res)['review_recall']['fold_mean'],
    '계약F1': summarize(res)['review_f1']['fold_mean'],
    'Dummy대비': summarize(res)['lift_over_dummy']['fold_mean'],
} for name, res in results.items()]).set_index('설정')
display(ladder.style.format('{:.3f}'))

print('Dummy는 정확도 0.49를 내지만 macro F1은 0.21이고 계약·질의검토 recall은 0.000입니다.')
print('세 클래스 중 둘을 아예 예측하지 않기 때문입니다. 정확도만 보면 "절반은 맞힌다"로')
print('읽히지만, 실무에서 정작 검토해야 할 조항은 하나도 못 찾습니다(§10.2, §11.9).')
print('LinearSVC와 검증 기반 추가 가중치는 현재 Logistic 기준선을 넘지 못했습니다.')

In [ ]:
display(Markdown('''## 3. 통제 비교 — 무엇이 효과였는가

그리드서치로 최고 조합을 뽑지 않습니다. 이 프로젝트의 질문은 "최고 점수가 몇 점인가"가
아니라 **"복잡하게 만든 값을 했는가"**이기 때문입니다(§9.3). 그리드서치는 점수를 주지만
어느 선택이 효과를 냈는지는 설명하지 못합니다.

**문서가 10개뿐이라 fold 간 분산이 큽니다.** 평균 차이가 fold별 편차에 비해 작으면
그것은 효과가 아니라 잡음입니다. 평균만 보면 이 구분이 안 보입니다.
'''))

def controlled(before, after, label, **kw):
    a, b = run_lodo(rows, before, **kw), run_lodo(rows, after, **kw)
    d = [y.macro_f1 - x.macro_f1 for x, y in zip(a, b)]
    spread = max(d) - min(d)
    return {
        '비교': label,
        '평균 차이': sum(d) / len(d),
        '최소': min(d), '최대': max(d), '편차 폭': spread,
        '우세': f"{sum(1 for x in d if x > 0)}/10",
        '판정': '잡음' if abs(sum(d) / len(d)) < spread / 4 else '효과',
    }

table = pd.DataFrame([
    controlled(WORD_BALANCED, CHAR_BALANCED, '표현: word -> char'),
    controlled(CHAR_UNWEIGHTED, CHAR_BALANCED, 'class_weight: 없음 -> balanced'),
]).set_index('비교')

nine = run_lodo(rows, CHAR_BALANCED, use_nine_documents=True)
d9 = [y.macro_f1 - x.macro_f1 for x, y in zip(results[CHAR_BALANCED.name], nine)]
table.loc['학습 문서: 8 -> 9'] = [
    sum(d9) / len(d9), min(d9), max(d9), max(d9) - min(d9),
    f"{sum(1 for x in d9 if x > 0)}/10",
    '잡음' if abs(sum(d9) / len(d9)) < (max(d9) - min(d9)) / 4 else '효과',
]
display(table.style.format({'평균 차이': '{:+.3f}', '최소': '{:+.3f}',
                            '최대': '{:+.3f}', '편차 폭': '{:.3f}'}))

display(Markdown('''**읽는 법**

- `class_weight`는 평균 +0.080에 9/10 우세로 **효과**입니다. 이 데이터의 기본값으로 둡니다.
- `학습 문서 8 -> 9`는 평균 +0.001이고 방향조차 일정하지 않아 **잡음**입니다.
  검증 문서 하나를 비워두는 손해가 잡음 수준이라는 뜻이고, 그래서 나중에 파인튜닝과
  나란히 놓기 위한 비교 가능성을 사실상 공짜로 얻습니다(§9.3).
- `표현: word -> char`는 평균 +0.020인데 편차 폭이 0.205입니다. **판정은 잡음**입니다.
  §9.2가 문자 n-gram을 "반드시 포함"이라고 둔 근거가 이 데이터에서는 그만큼 강하지
  않다는 뜻입니다. 다만 7/10 fold에서 우세하고 방향이 일정하므로, 표본이 늘면 효과로
  바뀔 여지가 있습니다. **지금 단계에서 단정하지 않고 기록해둡니다.**

두 효과의 크기 차이(0.080 vs 0.001)가 **어느 파라미터를 먼저 만질지**를 정해줍니다.
'''))

In [ ]:
display(Markdown('''## 4. macro F1 해부 — 왜 이 지표인가

`macro`는 **클래스별 F1을 구한 뒤 단순 평균**합니다. 건수를 전혀 반영하지 않습니다.
그래서 137건짜리 클래스와 25건짜리 클래스가 똑같이 1/3씩 들어갑니다.

mfds 문서로 보면 같은 예측이 지표에 따라 얼마나 달리 보이는지 드러납니다.
'''))

mfds = next(r for r in results[CHAR_BALANCED.name] if r.test_document == 'mfds_drug_ai_review')
mfds_fold = next(f for f in make_lodo_folds(rows) if f.test_document == 'mfds_drug_ai_review')
mfds_test = mfds_fold.split(rows)[2]
per_class = pd.DataFrame({'F1': mfds.per_class_f1})
per_class['평가 건수'] = pd.Series({
    l: sum(1 for r in mfds_test if r['primary_action'] == l)
    for l in mfds.per_class_f1
})
display(per_class.style.format({'F1': '{:.3f}'}))
print(f"macro F1 = 세 F1의 단순 평균  = {mfds.macro_f1:.3f}")
print(f"정확도   = 건수가 지배        = {mfds.accuracy:.3f}")
print("같은 예측인데 0.51과 0.70으로 갈립니다. 그래서 §10.2가 정확도를 보조 지표로만 씁니다.")
print("(micro F1은 단일 라벨 다중분류에서 정확도와 항상 같으므로 따로 내지 않습니다)\n")

display(Markdown('''### 진짜 문제는 불균형이 아니라 분포 이동입니다

전체 라벨은 50% / 24% / 26%로 최다·최소 2.1배입니다. 심한 불균형은 아닙니다.
**문제는 fold마다 학습 분포와 평가 분포가 어긋난다는 것**입니다.

`class_weight`는 불균형은 보정하지만 분포 이동은 못 고칩니다. 처방이 다릅니다.
'''))

shift = []
for fold in make_lodo_folds(rows):
    fit, _, test = fold.split(rows)
    row = {'평가 문서': fold.test_document}
    for part, part_rows in (('학습', fit), ('평가', test)):
        for label in mfds.per_class_f1:
            row[f'{part}·{label[:2]}'] = sum(
                1 for r in part_rows if r['primary_action'] == label) / len(part_rows)
    row['최대 어긋남'] = max(
        abs(row[f'학습·{l[:2]}'] - row[f'평가·{l[:2]}']) for l in mfds.per_class_f1)
    shift.append(row)
display(pd.DataFrame(shift).set_index('평가 문서')
        .sort_values('최대 어긋남', ascending=False)
        .style.format('{:.0%}'))
print('koen fold는 "통상수용 45%인 세상"을 배우고 나가서 "79%인 세상"을 만납니다.')
print('mfds에서 견적반영 F1이 0.196으로 낮았던 것도 이걸로 설명됩니다 — 25%를 기대했는데')
print('실제로는 13%였습니다. 개수는 거의 맞췄지만(예측 26 / 정답 25) 어느 것인지를 못 맞혔습니다.')

In [ ]:
display(Markdown('''## 5. 세 갈래 점수 — 점수에 무엇이 섞여 있는가

문서 간 표준 문구 반복은 **중복도 누수도 아닙니다.** 서로 다른 사업의 별개 요구사항이
문구를 공유하는 것이고, 실제 배포에서도 새 RFP에 같은 조항이 나옵니다. 모델이 알아보고
답하면 그건 정답입니다(결정 34).

남는 문제는 **점수 하나에 두 능력이 섞인다**는 것입니다. 그래서 전체 / 반복 제외 /
반복만으로 나눠 냅니다. 임계값 0.6 기준이며, 결론이 임계값에 크게 좌우되므로
(0.5면 16.3%, 0.8이면 4.9%) 항상 함께 적습니다.

**건수를 반드시 같이 봅니다.** 반복 부분집합은 fold에 따라 0건에서 43건까지입니다.
1건짜리에서 나온 0.000이나 1.000은 성능이 아니라 표본 크기의 산물입니다.
'''))

three = pd.DataFrame([{
    '평가 문서': r.test_document,
    '전체': r.macro_f1,
    '반복 제외': r.macro_f1_repeat_excluded,
    '반복만': r.macro_f1_repeat_only,
    '반복 건수': r.repeat_count,
    '노출률': r.repeat_exposure_rate,
} for r in results[CHAR_BALANCED.name]]).set_index('평가 문서')
display(three.style.format({'전체': '{:.3f}', '반복 제외': '{:.3f}',
                            '반복만': '{:.3f}', '노출률': '{:.1%}'}, na_rep='—'))

enough = three[three['반복 건수'] >= 10]
print(f"반복 건수 10건 이상인 fold {len(enough)}개에서만 비교합니다:")
print(f"  반복만 평균    {enough['반복만'].mean():.3f}")
print(f"  반복 제외 평균 {enough['반복 제외'].mean():.3f}")
print("\nmfds는 반복 문구에서 1.000, 나머지에서 0.460입니다. 전체 점수 0.506만 보고")
print('"처음 보는 RFP에 일반화된다"고 말하면 그 주장이 실제보다 강해 보입니다.')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
order = diag.sort_values('oracle').index
plot = pd.DataFrame({
    'Dummy(기준선)': diag['Dummy'],
    'char+balanced 정확도': pd.Series(
        {r.test_document: r.accuracy for r in results[CHAR_BALANCED.name]}),
}).loc[order]
plot.plot.barh(ax=axes[0], color=['#BAB0AC', '#4C78A8'])
axes[0].set(title='fold별 정확도와 그 fold의 기준선', xlabel='')
axes[0].legend(loc='lower right')

axes[1].scatter(diag['oracle'], diag['반복노출'], s=70, color='#E45756')
for name in diag.index:
    axes[1].annotate(name[:12], (diag.loc[name, 'oracle'], diag.loc[name, '반복노출']),
                     fontsize=8, xytext=(4, 4), textcoords='offset points')
axes[1].set(title=f'fold 난이도 vs 반복 노출률 (r = {r:.3f})',
            xlabel='oracle 기준선', ylabel='반복 노출률')
plt.tight_layout()

print('\n' + '=' * 66)
print('정리')
print(f"  - char TF-IDF + balanced: macro F1 {summarize(results[CHAR_BALANCED.name])['macro_f1']['fold_mean']:.3f}"
      f" (Dummy {summarize(results[DUMMY.name])['macro_f1']['fold_mean']:.3f})")
print('  - class_weight는 효과(+0.080), 학습 문서 8->9는 잡음(+0.001)')
print('  - word -> char는 평균 +0.020이나 편차가 커 아직 단정하지 않음')
print('  - 왼쪽 그림: 막대 길이가 아니라 회색과의 차이를 봐야 합니다')
print('  - 오른쪽 그림: 쉬운 fold가 공짜 점수도 더 받습니다(003과 006이 겹침)')